# DPC-UNet Fine-Tuning on Adversarial Examples

Fine-tunes the DPC-UNet denoising defense on adversarial images generated by FGSM, PGD,
and DeepFool against YOLOv8. Optimizes for pixel fidelity **and edge preservation** so
the denoised output retains spatial features YOLO depends on for detection.

**Before running this notebook:**
1. Locally: `python scripts/export_training_data.py` → creates `outputs/training_exports/<cycle_id>_training_data.zip`
   when run from a cycle signal, or `outputs/training_exports/training_data.zip` as the manual fallback
2. Upload the exported `*_training_data.zip` or `training_data.zip` file to your **Google Drive root**
3. In Colab: Runtime → Change runtime type → **T4 GPU**

**After training:**
- Download `dpc_unet_adversarial_finetuned.pt` from Drive
- Copy it to `YOLO-Bad-Triangle/`
- Set `DPC_UNET_CHECKPOINT_PATH=dpc_unet_adversarial_finetuned.pt` in `.env`
- Re-run the sweep to evaluate

In [ ]:
# Cell 1: Check GPU
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

In [ ]:
# Cell 2: Install missing dependencies (cv2 headless for Colab)
!pip install opencv-python-headless -q

In [ ]:
# Cell 3: Mount Google Drive and extract training data
import shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
zip_candidates = sorted(
    DRIVE_ROOT.glob('*_training_data.zip'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not zip_candidates:
    raise FileNotFoundError(
        'Upload <cycle_id>_training_data.zip to your Google Drive root first.\n'
        f'Expected under: {DRIVE_ROOT}'
    )
ZIP_PATH   = zip_candidates[0]
EXTRACT_DIR = Path('/content/training_data')
STAMP_FILE  = EXTRACT_DIR / '.zip_source'

# Re-extract if the zip changed since last run
if EXTRACT_DIR.exists():
    prior = STAMP_FILE.read_text().strip() if STAMP_FILE.exists() else ''
    if prior != ZIP_PATH.name:
        print(f'Zip changed ({prior!r} → {ZIP_PATH.name!r}), re-extracting...')
        shutil.rmtree(EXTRACT_DIR)

if not EXTRACT_DIR.exists():
    print(f'Extracting {ZIP_PATH.name}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    STAMP_FILE.write_text(ZIP_PATH.name)
else:
    print('Training data already extracted (same zip).')

print(f'Using training zip: {ZIP_PATH.name}')

clean_imgs    = sorted((EXTRACT_DIR / 'clean').glob('*.jpg'))
fgsm_imgs     = sorted((EXTRACT_DIR / 'adversarial/fgsm').glob('*.jpg'))
pgd_imgs      = sorted((EXTRACT_DIR / 'adversarial/pgd').glob('*.jpg'))
deepfool_imgs = sorted((EXTRACT_DIR / 'adversarial/deepfool').glob('*.jpg'))
blur_imgs     = sorted((EXTRACT_DIR / 'adversarial/blur').glob('*.jpg'))
checkpoint    = EXTRACT_DIR / 'checkpoint/dpc_unet_final_golden.pt'

print(f'Clean:    {len(clean_imgs)} images')
print(f'FGSM:     {len(fgsm_imgs)} images')
print(f'PGD:      {len(pgd_imgs)} images')
print(f'DeepFool: {len(deepfool_imgs)} images')
print(f'Blur:     {len(blur_imgs)} images')
print(f'Checkpoint exists: {checkpoint.exists()}')

In [ ]:
# Cell 4: DPC-UNet architecture (exact match to production wrapper)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


def sinusoidal_timestep_embedding(timesteps: Tensor, dim: int = 64) -> Tensor:
    if timesteps.ndim == 0:
        timesteps = timesteps.unsqueeze(0)
    timesteps = timesteps.float()
    half = dim // 2
    if half == 0:
        return timesteps.unsqueeze(-1)
    device = timesteps.device
    exponent = torch.arange(half, device=device, dtype=torch.float32) / max(half - 1, 1)
    freq = torch.exp(-torch.log(torch.tensor(10000.0, device=device)) * exponent)
    args = timesteps.unsqueeze(-1) * freq.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


def _num_groups(channels: int) -> int:
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class DPCResidualBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.mlp   = nn.Sequential(nn.SiLU(), nn.Linear(64, out_ch))
        self.conv1 = nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(_num_groups(out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(_num_groups(out_ch), out_ch)

    def forward(self, x: Tensor, t_embed: Tensor) -> Tensor:
        h = self.conv1(x)
        h = h + self.mlp(t_embed).unsqueeze(-1).unsqueeze(-1)
        h = F.silu(self.norm1(h))
        h = self.conv2(h)
        h = F.silu(self.norm2(h))
        return h


class DPCUNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.time_mlp   = nn.Sequential(nn.SiLU(), nn.Linear(64, 64))
        self.down1      = DPCResidualBlock(3, 32)
        self.down2      = DPCResidualBlock(32, 64)
        self.bottleneck = DPCResidualBlock(64, 128)
        self.up2_conv   = nn.Sequential(nn.Conv2d(128, 256, kernel_size=1), nn.PixelShuffle(2))
        self.up2_block  = DPCResidualBlock(128, 64)
        self.up1_conv   = nn.Sequential(nn.Conv2d(64, 128, kernel_size=1), nn.PixelShuffle(2))
        self.up1_block  = DPCResidualBlock(64, 32)
        self.final      = nn.Conv2d(32, 3, kernel_size=1)

    def forward(self, x: Tensor, timestep=50.0) -> Tensor:
        B = x.shape[0]
        if isinstance(timestep, (int, float)):
            t = torch.full((B,), float(timestep), device=x.device)
        else:
            t = timestep.to(x.device)
        t_embed = self.time_mlp(sinusoidal_timestep_embedding(t, dim=64))

        x1 = self.down1(x, t_embed)
        x2 = self.down2(F.avg_pool2d(x1, 2), t_embed)
        x3 = self.bottleneck(F.avg_pool2d(x2, 2), t_embed)

        up2 = self.up2_conv(x3)
        if up2.shape[-2:] != x2.shape[-2:]:
            up2 = F.interpolate(up2, size=x2.shape[-2:], mode='bilinear', align_corners=False)
        up2 = self.up2_block(torch.cat([up2, x2], dim=1), t_embed)

        up1 = self.up1_conv(up2)
        if up1.shape[-2:] != x1.shape[-2:]:
            up1 = F.interpolate(up1, size=x1.shape[-2:], mode='bilinear', align_corners=False)
        up1 = self.up1_block(torch.cat([up1, x1], dim=1), t_embed)
        return self.final(up1)


print(f'DPCUNet params: {sum(p.numel() for p in DPCUNet().parameters()):,}')

In [ ]:
# Cell 5: Dataset — loads (adversarial, clean) pairs with random crops and flips
import random
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader


class AdvPairDataset(Dataset):
    """
    Each item is (adversarial_tensor, clean_tensor), both [3, H, W] in [0, 1] RGB.
    Pairs are built from every (clean_image, attack_type) combination.
    """

    def __init__(self, pairs: list, patch_size: int = 256, augment: bool = True):
        self.pairs      = pairs   # list of (clean_path_str, adv_path_str, attack_name)
        self.patch_size = patch_size
        self.augment    = augment

    @classmethod
    def from_dirs(cls, clean_dir: Path, adv_dirs: dict, patch_size: int = 256, augment: bool = True):
        clean_paths = sorted(Path(clean_dir).glob('*.jpg'))
        pairs = []
        for clean_path in clean_paths:
            for attack_name, adv_dir in adv_dirs.items():
                adv_path = Path(adv_dir) / clean_path.name
                if adv_path.exists():
                    pairs.append((str(clean_path), str(adv_path), attack_name))
        print(f'  {len(clean_paths)} images × {len(adv_dirs)} attacks = {len(pairs)} pairs')
        return cls(pairs, patch_size=patch_size, augment=augment)

    def split(self, val_frac: float = 0.15, seed: int = 42):
        pairs = self.pairs.copy()
        random.Random(seed).shuffle(pairs)
        n_val = max(1, int(len(pairs) * val_frac))
        val_ds   = AdvPairDataset(pairs[:n_val],  self.patch_size, augment=False)
        train_ds = AdvPairDataset(pairs[n_val:],  self.patch_size, augment=True)
        return train_ds, val_ds

    def __len__(self) -> int:
        return len(self.pairs)

    def _load_and_crop(self, clean_path: str, adv_path: str):
        clean = cv2.imread(clean_path)
        adv   = cv2.imread(adv_path)
        if clean is None or adv is None:
            raise RuntimeError(f'Failed to load image pair: {clean_path}')

        p = self.patch_size
        h, w = clean.shape[:2]

        # Pad if smaller than patch
        if h < p or w < p:
            ph, pw = max(0, p - h), max(0, p - w)
            clean = cv2.copyMakeBorder(clean, 0, ph, 0, pw, cv2.BORDER_REFLECT)
            adv   = cv2.copyMakeBorder(adv,   0, ph, 0, pw, cv2.BORDER_REFLECT)
            h, w  = clean.shape[:2]

        if self.augment:
            y = random.randint(0, h - p)
            x = random.randint(0, w - p)
        else:
            y = (h - p) // 2
            x = (w - p) // 2

        return clean[y:y+p, x:x+p], adv[y:y+p, x:x+p]

    @staticmethod
    def _to_tensor(img_bgr: np.ndarray) -> torch.Tensor:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        return torch.from_numpy(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1)

    def __getitem__(self, idx: int):
        clean_path, adv_path, _ = self.pairs[idx]
        clean_bgr, adv_bgr = self._load_and_crop(clean_path, adv_path)

        if self.augment:
            if random.random() < 0.5:   # horizontal flip
                clean_bgr = cv2.flip(clean_bgr, 1)
                adv_bgr   = cv2.flip(adv_bgr,   1)
            if random.random() < 0.3:   # vertical flip
                clean_bgr = cv2.flip(clean_bgr, 0)
                adv_bgr   = cv2.flip(adv_bgr,   0)

        # input = adversarial, target = clean
        return self._to_tensor(adv_bgr), self._to_tensor(clean_bgr)

In [ ]:
# Cell 6: Loss functions
#
# L1 pixel loss:  forces pixel-level accuracy toward the clean image
# Sobel edge loss: forces edge/gradient preservation — directly targets the
#                  spatial features YOLO uses for detection

def sobel_edges(x: torch.Tensor) -> torch.Tensor:
    """Compute per-channel Sobel edge magnitude [B, C, H, W]."""
    kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                      dtype=torch.float32, device=x.device)
    ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                      dtype=torch.float32, device=x.device)
    C  = x.shape[1]
    kx = kx.view(1, 1, 3, 3).expand(C, 1, 3, 3)
    ky = ky.view(1, 1, 3, 3).expand(C, 1, 3, 3)
    gx = F.conv2d(x, kx, padding=1, groups=C)
    gy = F.conv2d(x, ky, padding=1, groups=C)
    return torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)


def denoising_loss(
    output: torch.Tensor,
    target: torch.Tensor,
    edge_weight: float = 0.15,
) -> tuple:
    """
    Returns (total_loss, pixel_loss_scalar, edge_loss_scalar).
    total = L1(output, target) + edge_weight * L1(edges(output), edges(target))
    """
    pixel = F.l1_loss(output, target)
    edge  = F.l1_loss(sobel_edges(output), sobel_edges(target))
    total = pixel + edge_weight * edge
    return total, pixel.item(), edge.item()

def _load_yolo_labels(label_file: Path, device) -> 'torch.Tensor | None':
    """Load YOLO-format label file. Returns (N,6) tensor [batch_idx, cls, cx, cy, w, h]."""
    rows = []
    for line in Path(label_file).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cls, cx, cy, w, h = (float(x) for x in parts)
        rows.append([0.0, cls, cx, cy, w, h])
    return torch.tensor(rows, dtype=torch.float32, device=device) if rows else None


In [ ]:
# Cell 7: Hyperparameters
# Round 2 settings: lower LR, shorter burst, targeted oversampling, detector-aligned loss

LEARNING_RATE  = 2e-5    # Lower than round 1 (5e-5) — continuing from converged checkpoint
BATCH_SIZE     = 16      # T4 handles 192x192 at batch 16 comfortably
NUM_EPOCHS     = 40      # Burst training — evaluate mAP50 after, extend if still improving
PATCH_SIZE     = 192     # Random crop size — smaller = faster per batch
TIMESTEP_MIN   = 5.0     # Minimum timestep sampled during training
TIMESTEP_MAX   = 100.0   # Maximum timestep (matches inference range)
EDGE_WEIGHT    = 0.25    # Weight for Sobel edge loss component
VAL_FRAC       = 0.10    # Fraction of pairs held out for validation
GRAD_CLIP      = 1.0     # Gradient clipping max norm
SEED           = 42
NUM_WORKERS    = 4

# Oversample deepfool_strong only — confirmed weak point (c_dog fails at ε=0.1)
# eot_pgd stays at 1× — low ROI for deterministic purifier (shadowboxing)
OVERSAMPLE     = {'deepfool_strong': 2}

# Detector-aligned loss: trains purifier against frozen YOLO's actual detection loss
# This directly moves mAP50 where pixel loss doesn't
LAMBDA_DET           = 0.15  # Weight of detector loss term (0=disabled)
DET_IMAGES_PER_EPOCH = 20    # Full images sampled per epoch for detector pass

# Always start fresh from golden checkpoint — do NOT resume the old 80-epoch run
# (that resume checkpoint has LR≈1e-7, effectively stale for new data)
FORCE_FRESH_START = True

SAVE_PATH      = '/content/drive/MyDrive/dpc_unet_adversarial_finetuned.pt'
RESUME_PATH    = '/content/drive/MyDrive/dpc_unet_training_resume_round2.pt'  # separate resume file

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True  # faster conv for fixed input size
print('Config set.')


In [ ]:
# Cell 8: Build dataset and dataloaders

ADV_DIRS = {}

# Retain deepfool and blur — multi-ε coverage + prevents catastrophic forgetting
# fgsm and pgd dropped — redundant with deepfool, waste small-model capacity
for attack in ['deepfool', 'blur']:
    d = EXTRACT_DIR / f'adversarial/{attack}'
    if d.is_dir() and list(d.glob('*.jpg')):
        ADV_DIRS[attack] = d
    else:
        print(f'WARNING: {attack} dir missing or empty, skipping')

# New attacks: deepfool at stronger ε=0.1, eot_pgd (1× — low ROI but included)
for attack in ['deepfool_strong', 'eot_pgd']:
    d = EXTRACT_DIR / f'adversarial/{attack}'
    if d.is_dir() and list(d.glob('*.jpg')):
        ADV_DIRS[attack] = d
        print(f'  Found new attack dir: {attack} ({len(list(d.glob("*.jpg")))} images)')
    else:
        print(f'WARNING: {attack} dir missing — zip may not include it')

# Clean→clean pairs: 'strength 0' curriculum entry — stabilizes identity mapping
clean_dir_path = EXTRACT_DIR / 'clean'
if clean_dir_path.is_dir() and list(clean_dir_path.glob('*.jpg')):
    ADV_DIRS['clean'] = clean_dir_path
    print(f'  Added clean→clean pairs: {len(list(clean_dir_path.glob("*.jpg")))} images')

print(f'Attack dirs: {list(ADV_DIRS.keys())}')

# Oversample deepfool_strong only
for name, n_times in OVERSAMPLE.items():
    if name in ADV_DIRS:
        for i in range(1, n_times):
            ADV_DIRS[f'{name}_rep{i}'] = ADV_DIRS[name]
        print(f'  Oversampling "{name}" x{n_times} (effective {n_times * 500} pairs)')

print('\nBuilding dataset...')
full_ds = AdvPairDataset.from_dirs(EXTRACT_DIR / 'clean', ADV_DIRS,
                                   patch_size=PATCH_SIZE, augment=True)
train_ds, val_ds = full_ds.split(val_frac=VAL_FRAC, seed=SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds):>5} pairs  ({len(train_loader)} batches/epoch)')
print(f'Val:   {len(val_ds):>5} pairs  ({len(val_loader)} batches)')


In [ ]:
# Cell 9: Load checkpoint and set up optimizer (fresh-start-aware)

model = DPCUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
)

scaler = torch.cuda.amp.GradScaler()  # AMP scaler
resume_file = Path(RESUME_PATH)
start_epoch = 1
best_val_loss = float('inf')
history = {'train': [], 'val': [], 'pixel': [], 'edge': []}

if not FORCE_FRESH_START and resume_file.is_file():
    print(f'Resume checkpoint found: {RESUME_PATH}')
    ckpt = torch.load(str(resume_file), map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    if 'scaler' in ckpt:
        scaler.load_state_dict(ckpt['scaler'])
    start_epoch   = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    history       = ckpt['history']
    print(f'  Resuming from epoch {start_epoch}/{NUM_EPOCHS}  (best val so far: {best_val_loss:.4f})')
else:
    # Fresh start — load current golden checkpoint weights
    # FORCE_FRESH_START=True ensures we don't accidentally resume the old 80-epoch run
    state_dict = torch.load(str(checkpoint), map_location='cpu', weights_only=True)
    model.load_state_dict(state_dict, strict=True)
    reason = 'FORCE_FRESH_START=True' if FORCE_FRESH_START else 'no resume file'
    print(f'Fresh start ({reason}) — loaded: {checkpoint.name}')

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Optimizer: Adam lr={LEARNING_RATE}, cosine decay to 1e-7 over {NUM_EPOCHS} epochs')


In [ ]:
# Cell 9b: Frozen YOLO for detector-aligned loss
from types import SimpleNamespace

yolo_det = None
full_det_pairs = []
labels_dir_nb = EXTRACT_DIR / 'labels'

if LAMBDA_DET > 0 and labels_dir_nb.is_dir():
    from ultralytics import YOLO as _YOLO
    # Try custom model from zip, fall back to yolov8n (auto-downloaded by ultralytics)
    _ckpt = EXTRACT_DIR / 'checkpoint' / 'yolo26n.pt'
    if not _ckpt.exists():
        _ckpt = Path('yolov8n.pt')
    yolo_det = _YOLO(str(_ckpt)).model
    # Build args SimpleNamespace with required loss weights
    _required_hyp = {'box': 7.5, 'cls': 0.5, 'dfl': 1.5}
    _existing = {}
    if hasattr(yolo_det, 'args') and yolo_det.args is not None:
        if isinstance(yolo_det.args, dict):
            _existing = yolo_det.args
        elif hasattr(yolo_det.args, '__dict__'):
            _existing = vars(yolo_det.args)
    yolo_det.args = SimpleNamespace(**{**_existing, **_required_hyp})
    # Move to device FIRST — criterion captures device via next(model.parameters()).device
    yolo_det = yolo_det.to(device)
    yolo_det.criterion = yolo_det.init_criterion()
    yolo_det.train()
    for p in yolo_det.parameters():
        p.requires_grad_(False)
    for cp, ap, _ in full_ds.pairs:
        if (labels_dir_nb / (Path(cp).stem + '.txt')).exists():
            full_det_pairs.append((cp, ap))
    print(f'YOLO frozen. {len(full_det_pairs)} images eligible for detector loss.')
elif LAMBDA_DET > 0:
    print('WARNING: labels dir not found in zip — detector loss disabled')
else:
    print('Detector loss disabled (LAMBDA_DET=0)')


In [ ]:
# Cell 10: Training loop
from tqdm.notebook import tqdm

for epoch in range(start_epoch, NUM_EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    ep_loss = ep_pixel = ep_edge = 0.0

    bar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{NUM_EPOCHS} train', leave=False)
    for adv_imgs, clean_imgs in bar:
        adv_imgs   = adv_imgs.to(device, non_blocking=True)
        clean_imgs = clean_imgs.to(device, non_blocking=True)

        t = random.uniform(TIMESTEP_MIN, TIMESTEP_MAX)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda'):
            output = model(adv_imgs, timestep=t)
            output = torch.clamp(output, 0.0, 1.0)
            loss, pixel_l, edge_l = denoising_loss(output, clean_imgs, EDGE_WEIGHT)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        ep_loss  += loss.item()
        ep_pixel += pixel_l
        ep_edge  += edge_l
        bar.set_postfix(loss=f'{loss.item():.4f}', t=f'{t:.0f}')

    train_loss  = ep_loss  / len(train_loader)
    train_pixel = ep_pixel / len(train_loader)
    train_edge  = ep_edge  / len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for adv_imgs, clean_imgs in val_loader:
            adv_imgs   = adv_imgs.to(device, non_blocking=True)
            clean_imgs = clean_imgs.to(device, non_blocking=True)
            output = torch.clamp(model(adv_imgs, timestep=50.0), 0.0, 1.0)
            loss, _, _ = denoising_loss(output, clean_imgs, EDGE_WEIGHT)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    # ── Detector alignment pass (one update per epoch) ─────────────────────
    det_loss_val = 0.0
    if yolo_det is not None and full_det_pairs:
        sampled = random.sample(full_det_pairs,
                                min(DET_IMAGES_PER_EPOCH, len(full_det_pairs)))
        optimizer.zero_grad(set_to_none=True)
        n_ok = 0
        for clean_path, adv_path in sampled:
            adv_bgr = cv2.imread(adv_path)
            if adv_bgr is None:
                continue
            adv_t = (torch.from_numpy(
                cv2.cvtColor(cv2.resize(adv_bgr, (640, 640)),
                             cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            ).permute(2, 0, 1).unsqueeze(0).to(device))
            lbl_path = labels_dir_nb / (Path(clean_path).stem + '.txt')
            lbl_t = _load_yolo_labels(lbl_path, device)
            if lbl_t is None:
                continue
            model.train()
            purified = torch.clamp(model(adv_t, timestep=50.0), 0.0, 1.0)
            preds = yolo_det(purified)
            det_loss, _ = yolo_det.loss(
                {'img': purified, 'batch_idx': lbl_t[:, 0],
                 'cls': lbl_t[:, 1], 'bboxes': lbl_t[:, 2:]},
                preds=preds,
            )
            (det_loss.sum() * LAMBDA_DET / len(sampled)).backward()
            det_loss_val += det_loss.sum().item()
            n_ok += 1
        if n_ok > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            det_loss_val /= n_ok

    scheduler.step()

    history['train'].append(train_loss)
    history['val'].append(val_loss)
    history['pixel'].append(train_pixel)
    history['edge'].append(train_edge)

    # Save best model
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)

    # Save resume checkpoint every epoch (overwrites previous)
    torch.save({
        'epoch':         epoch,
        'model':         model.state_dict(),
        'optimizer':     optimizer.state_dict(),
        'scheduler':     scheduler.state_dict(),
        'scaler':        scaler.state_dict(),
        'best_val_loss': best_val_loss,
        'history':       history,
    }, RESUME_PATH)

    lr_now = scheduler.get_last_lr()[0]
    print(
        f'Epoch {epoch:02d}/{NUM_EPOCHS}  '
        f'train={train_loss:.4f} (px={train_pixel:.4f} edge={train_edge:.4f})  '
        f'val={val_loss:.4f}  '
        + (f'det={det_loss_val:.4f}  ' if yolo_det is not None else '')
        + f'lr={lr_now:.1e}'
        + ('  ← best' if is_best else '')
    )

print(f'\nTraining complete. Best val loss: {best_val_loss:.4f}')
print(f'Best checkpoint: {SAVE_PATH}')
print(f'Resume checkpoint: {RESUME_PATH}  (safe to delete after downloading best)')


In [ ]:
# Cell 11: Training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(history['train']) + 1)
ax1.plot(epochs, history['train'], label='train')
ax1.plot(epochs, history['val'],   label='val')
ax1.set_title('Total Loss (L1 + edge)')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, history['pixel'], label='pixel L1')
ax2.plot(epochs, history['edge'],  label='edge (Sobel)')
ax2.set_title('Train Loss Components')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Visual sanity check — Clean vs Adversarial vs Denoised
import matplotlib.pyplot as plt

# Reload best checkpoint for visualization
model.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
model.eval()

# Pick one example per attack type
attack_samples = {}
for clean_path, adv_path, attack_name in full_ds.pairs:
    if attack_name not in attack_samples:
        attack_samples[attack_name] = (clean_path, adv_path)
    if len(attack_samples) == 3:
        break

def bgr_to_display(bgr, max_side=384):
    h, w = bgr.shape[:2]
    if max(h, w) > max_side:
        scale = max_side / max(h, w)
        bgr   = cv2.resize(bgr, (int(w * scale), int(h * scale)))
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

n = len(attack_samples)
fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
if n == 1:
    axes = [axes]

for row, (attack_name, (clean_path, adv_path)) in enumerate(attack_samples.items()):
    clean_bgr = cv2.imread(clean_path)
    adv_bgr   = cv2.imread(adv_path)

    # Run model on full image (no crop)
    adv_rgb = cv2.cvtColor(adv_bgr, cv2.COLOR_BGR2RGB)
    adv_t   = torch.from_numpy(adv_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out_t = torch.clamp(model(adv_t, timestep=50.0), 0.0, 1.0)
    denoised_rgb = (out_t.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    denoised_bgr = cv2.cvtColor(denoised_rgb, cv2.COLOR_RGB2BGR)

    axes[row][0].imshow(bgr_to_display(clean_bgr))
    axes[row][0].set_title('Clean (target)')
    axes[row][0].axis('off')

    axes[row][1].imshow(bgr_to_display(adv_bgr))
    axes[row][1].set_title(f'Adversarial ({attack_name})')
    axes[row][1].axis('off')

    axes[row][2].imshow(bgr_to_display(denoised_bgr))
    axes[row][2].set_title('DPC-UNet Denoised')
    axes[row][2].axis('off')

plt.suptitle('Fine-tuned DPC-UNet: Adversarial Purification', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Next Steps

1. **Download** `dpc_unet_adversarial_finetuned.pt` from your Google Drive
2. **Copy** it into the `YOLO-Bad-Triangle/` project root
3. **Update** `.env`:
   ```
   DPC_UNET_CHECKPOINT_PATH=dpc_unet_adversarial_finetuned.pt
   ```
4. **Re-run the sweep** to evaluate the fine-tuned model:
   ```bash
   PYTHONPATH=src ./.venv/bin/python scripts/sweep_and_report.py \
     --attacks fgsm,pgd,deepfool,blur \
     --defenses c_dog,c_dog_ensemble,median_preprocess \
     --preset full \
     --workers 1 \
     --validation-enabled
   ```
5. **Compare** the new `framework_run_report.md` against the baseline sweep

### What to look for
- `c_dog` recovery should move from **−433%** (original) toward **positive** values
- Detection counts for defended runs should be closer to the attacked (not defended) baseline
- `deepfool + c_dog` is the hardest case — it should show the biggest relative improvement
  since deepfool was in the training data and caused the most damage in the original sweep

### If recovery is still poor
- Try increasing `NUM_EPOCHS` to 100
- Try reducing `TIMESTEP_MAX` to 50 (focus on the t=50 inference setting)
- Try increasing `EDGE_WEIGHT` to 0.3 to prioritize structural preservation more aggressively
- Consider adding a **YOLO feature loss**: run the YOLO backbone on `output` and `clean`,
  penalize L1 distance between intermediate feature maps — this directly optimizes for
  detection-relevant representations

## Feature-Matching Loss Ablation (YOLO-aware)

This section adds a YOLO feature-matching loss term to the existing pixel+edge objective.

Two conditions are compared:

1. `pixel + edge` (current baseline)
2. `pixel + edge + feature` (new)

The feature term uses frozen YOLO backbone activations to preserve detector-relevant structure in denoised outputs.

In [ ]:
# Cell 14: Feature-loss helpers (inlined — no local lab dependency)
from dataclasses import dataclass
from typing import Any

# ── CompositeLossWeights ───────────────────────────────────────────────────
@dataclass(frozen=True)
class CompositeLossWeights:
    pixel_weight: float = 1.0
    edge_weight: float = 0.15
    feature_weight: float = 0.1

# ── FeatureLossConfig ──────────────────────────────────────────────────────
@dataclass(frozen=True)
class FeatureLossConfig:
    layer_names: tuple
    reduction: str = "mean"

# ── YOLOFeatureExtractor ───────────────────────────────────────────────────
class YOLOFeatureExtractor:
    def __init__(self, yolo_model: Any, *, config: FeatureLossConfig) -> None:
        self.config = config
        model_obj = getattr(yolo_model, "model", yolo_model)
        self.model = getattr(model_obj, "model", model_obj)
        if not isinstance(self.model, torch.nn.Module):
            raise TypeError("YOLOFeatureExtractor requires a torch.nn.Module.")
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad_(False)
        self._hooks = []
        self._features = {}
        self._register_hooks()

    def _register_hooks(self) -> None:
        by_name = dict(self.model.named_modules())
        for name in self.config.layer_names:
            module = by_name.get(name)
            if module is None:
                raise ValueError(f"Feature layer '{name}' not found in YOLO model.")
            def _capture(_m, _i, output, *, key=name):
                if isinstance(output, torch.Tensor):
                    self._features[key] = output
            self._hooks.append(module.register_forward_hook(_capture))

    def close(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()
        self._features.clear()

    def extract(self, x: torch.Tensor) -> dict:
        self._features = {}
        _ = self.model(x)
        return dict(self._features)

# ── yolo_feature_matching_loss ─────────────────────────────────────────────
def yolo_feature_matching_loss(*, extractor, denoised, clean):
    den_feat   = extractor.extract(denoised)
    clean_feat = extractor.extract(clean)
    losses = []
    for name in extractor.config.layer_names:
        if name in den_feat and name in clean_feat:
            losses.append(F.l1_loss(den_feat[name], clean_feat[name].detach()))
    if not losses:
        return denoised.new_zeros(())
    stacked = torch.stack(losses)
    return stacked.mean() if extractor.config.reduction == "mean" else stacked.sum()

# ── composite_denoising_loss ───────────────────────────────────────────────
def composite_denoising_loss(*, denoised, clean, yolo_feature_loss, weights):
    pixel_loss   = F.l1_loss(denoised, clean)
    edge_loss    = F.l1_loss(sobel_edges(denoised), sobel_edges(clean))
    feature_loss = yolo_feature_loss if yolo_feature_loss is not None else denoised.new_zeros(())
    total = (weights.pixel_weight * pixel_loss
           + weights.edge_weight  * edge_loss
           + weights.feature_weight * feature_loss)
    return total, {
        "pixel_loss":   float(pixel_loss.detach()),
        "edge_loss":    float(edge_loss.detach()),
        "feature_loss": float(feature_loss.detach()),
        "total_loss":   float(total.detach()),
    }

# ── YOLO feature extractor setup ───────────────────────────────────────────
FEATURE_WEIGHT = 0.10
FEATURE_LAYERS = ('model.4', 'model.6', 'model.9')

# YOLO feature extractor skipped — not used in training loop, saves load time
feature_extractor = None
print('Feature extractor skipped (not used in training loop).')

def denoising_loss_with_feature(output: torch.Tensor, clean: torch.Tensor):
    feat_loss = (
        yolo_feature_matching_loss(extractor=feature_extractor, denoised=output, clean=clean)
        if feature_extractor is not None else None
    )
    return composite_denoising_loss(
        denoised=output, clean=clean,
        yolo_feature_loss=feat_loss,
        weights=CompositeLossWeights(
            pixel_weight=1.0, edge_weight=EDGE_WEIGHT, feature_weight=FEATURE_WEIGHT,
        ),
    )

print('Feature-loss helpers ready.')


In [ ]:
# Cell 15: Mini ablation step (pixel+edge vs pixel+edge+feature)
# Run this before full retraining to verify loss numerics and logging.

model.eval()
adv_imgs, clean_imgs = next(iter(train_loader))
adv_imgs = adv_imgs.to(device, non_blocking=True)
clean_imgs = clean_imgs.to(device, non_blocking=True)

with torch.no_grad():
    out = torch.clamp(model(adv_imgs, timestep=50), 0.0, 1.0)

base_total, base_terms = composite_denoising_loss(
    denoised=out,
    clean=clean_imgs,
    yolo_feature_loss=None,
    weights=CompositeLossWeights(pixel_weight=1.0, edge_weight=EDGE_WEIGHT, feature_weight=0.0),
)
feat_total, feat_terms = denoising_loss_with_feature(out, clean_imgs)

print('Baseline terms:', base_terms)
print('Feature terms: ', feat_terms)
print(f"Delta total loss (feature - baseline): {float(feat_total.item() - base_total.item()):.6f}")

# In the main training loop, replace:
#   loss, pixel_l, edge_l = denoising_loss(output, clean_imgs, EDGE_WEIGHT)
# with:
#   loss, terms = denoising_loss_with_feature(output, clean_imgs)
#   pixel_l = terms['pixel_loss']; edge_l = terms['edge_loss']
# and optionally track terms['feature_loss'] in history.
